In [1]:
import torch
import torch.nn as nn
import numpy as np
import torch.nn.functional as F
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")

print("Path to dataset files:", path)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
path=os.path.join(path,'PetImages')

In [3]:
import os
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from Preprocessing import AutoTokenizer
tokenizer=AutoTokenizer()
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.samples = []
        
        for class_name in self.classes:
            class_path = os.path.join(root_dir, class_name)
            for img_name in os.listdir(class_path):
                self.samples.append((os.path.join(class_path, img_name), self.class_to_idx[class_name]))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        return image, label

# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Load dataset
dataset = CustomImageDataset(root_dir=path, transform=transform)

# Split dataset into train and validation sets
from sklearn.model_selection import train_test_split

train_samples, val_samples = train_test_split(dataset.samples, 
                                            test_size=0.2, 
                                            random_state=42,
                                            stratify=[s[1] for s in dataset.samples])

# Create custom datasets for train and validation
train_dataset = CustomImageDataset(root_dir=path, 
                                 transform=transform)
val_dataset = CustomImageDataset(root_dir=path, 
                               transform=transform)

# Override samples in the new datasets
train_dataset.samples = train_samples
val_dataset.samples = val_samples

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
#device=torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
device=torch.device('cpu')

In [4]:
from models import Transformer
from torch.optim import Adam
from tqdm import tqdm 
class binary_classifier(nn.Module):
    def __init__(self,input_dim=None,n_labels=None):
        super().__init__()
        self.n_labels=n_labels
        self.input_dim=input_dim
        self.transformer=Transformer(input_dim=input_dim)
        self.classifier=nn.Sequential(
            nn.Linear(588,32),
            nn.Dropout(0.3),
            nn.ReLU(),
            nn.Linear(32,16),
            nn.ReLU(),
            nn.Linear(16,n_labels)
        )
    def forward(self,x):
        x=self.transformer(x)
        return self.classifier(x)
model=binary_classifier(input_dim=588,n_labels=2)
optimizer=Adam(params=model.parameters(),lr=0.0001,weight_decay=0.01)
criterion=torch.nn.CrossEntropyLoss()
num_epochs=10
for epoch in range(num_epochs):
    total_true=0
    y_true=[]
    y_pred=[]
    pbar= tqdm(train_loader,desc=f"Epoch{epoch}/{num_epochs}",leave=True)
    for image , label in pbar:
        image=image.to(device)
        label=label.to(device)
        token=tokenizer.tokenize(img=image)
        token=token.to(device)
        label=label.to(device)
        optimizer.zero_grad()
        output=model(token)
        _,predicted=torch.max(output)
        loss=criterion(label,predicted)
        loss.backward()
        optimizer.step()
    

Epoch0/10:   0%|          | 0/625 [00:00<?, ?it/s]


torch.Size([32, 256, 588])


EinopsError:  Error while processing rearrange-reduction pattern "b n_patch (n_layers dim_k) -> b n_layers n_patch dim_k".
 Input tensor shape: torch.Size([32, 256, 588]). Additional info: {'n_layers': 8}.
 Shape mismatch, can't divide axis of length 588 in chunks of 8